In [1]:
import numpy as np
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("postgresql+psycopg2://postgres@127.0.0.1:5432/trading")

def load_bars(tf):
    b = pd.read_sql(
        "SELECT ts_utc, open, high, low, close, atr_14, session FROM fx_bars "
        "WHERE pair = 'EURUSD' AND timeframe = %(tf)s ORDER BY ts_utc",
        engine, params={"tf": tf}, index_col="ts_utc",
    )
    b.index = pd.to_datetime(b.index, utc=True)
    return b

b15 = load_bars("15m")
b60 = load_bars("1h")
b15 = b15[b15.index < "2022-01-01"]     # BUILD PERIOD ONLY (2015-21); 2024-25 test stays locked
b60 = b60[b60.index < "2022-01-01"]
print(len(b15), len(b60))

174071 43527


In [2]:
import numpy as np
import pandas as pd

PIP = 0.0001
COST_PIPS = 1.4          # spread + slippage, round trip (from the specs)

def simulate_trade(bars, entry_i, side, stop, target, max_bars=None, exit_time=None):
    """
    bars:      DataFrame with open/high/low/close, index = bar start time (UTC)
    entry_i:   integer position of the bar we ENTER on (fill at its open)
    side:      +1 long, -1 short
    stop, target: absolute prices
    max_bars:  close at that bar's close if still open after this many bars
    exit_time: 'HH:MM' UTC, close at the open of the first bar at/after this time
    Returns a dict describing the trade, with P&L in pips after costs.
    """
    entry_px = bars["open"].iat[entry_i]
    n = len(bars)
    for k, i in enumerate(range(entry_i, n)):
        t = bars.index[i]
        o, h, l, c = bars["open"].iat[i], bars["high"].iat[i], bars["low"].iat[i], bars["close"].iat[i]

        # time exit: leave at this bar's open
        if exit_time is not None and k > 0 and t.strftime("%H:%M") >= exit_time:
            exit_px, reason = o, "time"
            break

        if side == 1:
            hit_stop, hit_tgt = l <= stop, h >= target
        else:
            hit_stop, hit_tgt = h >= stop, l <= target

        if hit_stop:                       # checked FIRST: worst case if both touched
            # if the bar opened beyond the stop (a gap), we fill at the open, not the stop
            gap = (o < stop) if side == 1 else (o > stop)
            exit_px, reason = (o if gap else stop), "stop"
            break
        if hit_tgt:
            exit_px, reason = target, "target"
            break
        if max_bars is not None and k + 1 >= max_bars:
            exit_px, reason = c, "max_bars"
            break
    else:
        exit_px, reason, i = bars["close"].iat[n - 1], "end_of_data", n - 1

    gross = side * (exit_px - entry_px) / PIP
    return {
        "entry_time": bars.index[entry_i], "exit_time": bars.index[i],
        "side": side, "entry": entry_px, "exit": exit_px, "reason": reason,
        "bars_held": i - entry_i + 1,
        "pips_gross": gross, "pips_net": gross - COST_PIPS,
        "risk_pips": abs(entry_px - stop) / PIP,
    }

In [3]:
def run_strategy_b(b60, band=2.0, stop_mult=1.5, max_bars=8, hours=range(0, 7)):
    trades, skipped = [], {"no_data": 0, "high_vol": 0, "target_wrong_side": 0}
    upper = b60["ma20"] + band * b60["sd20"]
    lower = b60["ma20"] - band * b60["sd20"]
    i, n = 0, len(b60)
    while i < n - 1:
        t = b60.index[i]
        c = b60["close"].iat[i]
        side = 1 if c < lower.iat[i] else (-1 if c > upper.iat[i] else 0)
        if side == 0 or t.hour not in hours:                  # signal bar must START in 00:00-06:00 UTC
            i += 1; continue
        atr, cap, ma = b60["atr_14"].iat[i], b60["atr_cap"].iat[i], b60["ma20"].iat[i]
        if np.isnan(atr) or np.isnan(cap) or np.isnan(ma):
            skipped["no_data"] += 1; i += 1; continue
        if atr > cap:
            skipped["high_vol"] += 1; i += 1; continue
        entry_i = i + 1                                # fill at the NEXT bar's open
        entry_px = b60["open"].iat[entry_i]
        target = ma                                    # fixed at signal time
        if side * (target - entry_px) <= 0:            # already snapped back by the open
            skipped["target_wrong_side"] += 1; i += 1; continue
        stop = entry_px - side * stop_mult * atr
        tr = simulate_trade(b60, entry_i, side, stop, target, max_bars=max_bars)
        trades.append(tr)
        i = b60.index.get_loc(tr["exit_time"]) + 1     # one position at a time
    return pd.DataFrame(trades), skipped


In [4]:
b60 = b60.copy()
b60["ma20"] = b60["close"].rolling(20).mean()
b60["sd20"] = b60["close"].rolling(20).std()
b60["atr_cap"] = b60["atr_14"].shift(1).rolling(60 * 24, min_periods=500).quantile(0.80)

sessions = {"asia": range(0, 7), "london": range(7, 12), "overlap": range(12, 16),
            "newyork": range(16, 21), "late": range(21, 24)}
rows = []
for name, hrs in sessions.items():
    tr, _ = run_strategy_b(b60, band=1.5, stop_mult=2.0, hours=hrs)
    R = tr["pips_net"] / tr["risk_pips"]
    Rg = tr["pips_gross"] / tr["risk_pips"]
    rows.append({"session": name, "trades": len(tr), "win_rate": (tr.pips_net > 0).mean(),
                 "avg_R": R.mean(), "avg_R_gross": Rg.mean(),
                 "cost_R": (Rg - R).mean(), "se": R.std() / np.sqrt(len(tr))})
pd.DataFrame(rows).round(3)

,session,trades,win_rate,avg_R,avg_R_gross,cost_R,se
0,asia,532,0.530,-0.032,0.032,0.064,0.035
1,london,1529,0.513,-0.065,0.006,0.071,0.025
2,overlap,1505,0.508,-0.069,-0.011,0.058,0.022
3,newyork,783,0.473,-0.068,-0.018,0.050,0.020
4,late,224,0.504,-0.029,0.020,0.048,0.035
